## Exploratory Data Analysis

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D
import math

In [3]:
#payload composition & class balance for the LNP dataset


PAYLOAD_MAP = {0: "mRNA", 1: "siRNA"}
LABEL_MAP = {0: "Not formed", 1: "Formed"}

df = pd.read_csv("./datasets/processed_data.csv", usecols=["label", "Payload"])
df["Payload"] = df["Payload"].map(PAYLOAD_MAP)
df["label"] = df["label"].map(LABEL_MAP)

table = df.groupby("Payload")["label"].value_counts().unstack()
table["Total"] = table.sum(axis=1)
table = table.reindex(["mRNA", "siRNA"])
table.loc["Overall"] = table.sum()

pct_formed = (table["Formed"] / table["Total"] * 100).round(1)
table["% Formed"] = pct_formed

table.to_csv("./analysis/Payload_class_distribution.csv")
print(table)

# Main-text sentence
n = len(df)
n_mrna, n_sirna = table.loc["mRNA", "Total"], table.loc["siRNA", "Total"]
n_formed, n_not_formed = table.loc["Overall", "Formed"], table.loc["Overall", "Not formed"]
pct_mrna = round(n_mrna / n * 100, 1)
pct_sirna = round(n_sirna / n * 100, 1)
pct_formed = round(n_formed / n * 100, 1)

print(f"\nDataset: {n} LNP formulations — "
      f"{n_mrna} mRNA ({pct_mrna}%), {n_sirna} siRNA ({pct_sirna}%). "
      f"Overall class balance: {n_formed} formed : {n_not_formed} not formed "
      f"({pct_formed}% formed).")

label    Formed  Not formed  Total  % Formed
Payload                                     
mRNA       2014         772   2786      72.3
siRNA       947         269   1216      77.9
Overall    2961        1041   4002      74.0

Dataset: 4002 LNP formulations — 2786 mRNA (69.6%), 1216 siRNA (30.4%). Overall class balance: 2961 formed : 1041 not formed (74.0% formed).


In [4]:
# component identities, molar/mass ratio ranges (mRNA/siRNA)
df = pd.read_csv("./datasets/processed_data.csv")
df["PEGChain"] = df["PEGChain"].str.strip()
df["PEG-lipid"] = df["PEGChain"] + "PEG" + (df["PEG MW"] // 1000).astype(str) + "K"
df["Payload"] = df["Payload"].map(PAYLOAD_MAP)

components = {
    "Ionizable lipid": "Lipomer",
    "Cholesterol": "Cholesterol",
    "PEG-lipid": "PEG-lipid",
    "Helper lipid": "HelperLipid",
}
molar_cols = ["Lipomer Mole %", "Cholesterol Mole %", "PEG Mole %", "Helper Lipid Mole %"]

# --- Unique component counts (main text + bar plot) ---
unique_counts = pd.DataFrame()
for label, col in components.items():
    unique_counts[label] = df.groupby("Payload")[col].nunique()

overall_counts = df[list(components.values())].nunique()
unique_counts.loc["Overall"] = overall_counts.values

unique_counts.to_csv("./analysis/table_S2a_unique_component_counts.csv")
print(unique_counts)

         Ionizable lipid  Cholesterol  PEG-lipid  Helper lipid
Payload                                                       
mRNA                 136           18          3            21
siRNA                 77            9          7            29
Overall              187           25          7            40


In [5]:
# --- Molecule identity + count table, one CSV per component ---
for label, col in components.items():
    counts = df.groupby(col)["Payload"].value_counts().unstack(fill_value=0)
    counts["Total"] = counts.sum(axis=1)
    counts = counts.sort_values("Total", ascending=False)
    counts.to_csv(f"./analysis/table_S2b_{col}_counts.csv")
counts

Payload,mRNA,siRNA,Total
HelperLipid,,,
DOPE,1780,121,1901
DSPC,115,676,791
DOTAP,352,12,364
18:1 Lyso PC,118,76,194
DOTMA,140,12,152
DOPC,45,12,57
18:1 Dimethyl PE,32,12,44
18:1 CAP PE,32,12,44
18:1 Monomethyl PE,32,12,44


In [16]:
def draw_grid(mols, legends, mols_per_row=6, sub_size=(200, 200)):
    n_rows = math.ceil(len(mols) / mols_per_row)
    drawer = rdMolDraw2D.MolDraw2DSVG(sub_size[0] * mols_per_row, sub_size[1] * n_rows,
                                       sub_size[0], sub_size[1])
    drawer.DrawMolecules(mols, legends=legends)
    drawer.FinishDrawing()
    return drawer.GetDrawingText()

smiles = pd.read_csv("./datasets/Smiles_complete.csv", encoding="utf-8-sig").set_index("Name")["SMILES"]

for label, col in components.items():
    counts = df.groupby(col)["Payload"].value_counts().unstack(fill_value=0)
    counts["Total"] = counts.sum(axis=1)
    counts = counts.sort_values("Total", ascending=False)

    names = [name for name in counts.index if name in smiles]
    mols = [Chem.MolFromSmiles(smiles[name]) for name in names]
    legends = [f"{name} (mRNA={counts.loc[name, 'mRNA']}, siRNA={counts.loc[name, 'siRNA']})" for name in names]

    svg = draw_grid(mols, legends)
    with open(f"./analysis/figS2_{col}_structures.svg", "w") as f:
        f.write(svg)